<a href="https://colab.research.google.com/github/melissa-04/melisayla-biyoinformatik/blob/main/notebooks/depmap/01_veri_mutfagi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Veri mutfağı: DepMap CRISPR ekranları

Dördüncü proje bir boru hattı projesi değil: ham okuma, hizalama, sayım yok. Elimizde hazır bir tablo var ve iş, o tabloya doğru soruları sormak.

Veri, Broad Institute'un DepMap projesinden geliyor. Yüzlerce kanser hücre hattında genom çapında CRISPR-Cas9 nakavt ekranları yapılmış: her hatta her gen tek tek devre dışı bırakılmış ve hücrelerin çoğalmasına ne olduğu ölçülmüş. Sonuç, hücre hattı × gen boyutunda bir **bağımlılık skoru** matrisidir.

Bu defterin işi mutfak: dosyaları oku, tanı, ders alt kümesini çıkar. DepMap dosyaları çeyrek çeyrek güncellendiği için sürümü sabitlemek teknik bir zorunluluk — aynı analiz bir yıl sonra farklı sayı verebilir.

## 1. Drive'ı bağlamak

Dosyalar Drive'da `melisa-ile-biyoinformatik` klasöründe duruyor. `drive.mount` Drive'ı Colab'ın dosya sistemine bağlar; ilk çalıştırmada bir izin penceresi açılır ve hesabınızı seçmeniz istenir. Bağlandıktan sonra Drive içeriği `/content/drive/MyDrive/` altından normal klasör gibi okunur — 430 MB'lık dosyayı Colab'a ayrıca yüklemeye gerek kalmaz.

`pd.read_csv` büyük dosyada bir-iki dakika sürebilir; `index_col=0` ilk sütunu (hücre hattı kimliği) satır adı yapar.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

KLASOR = '/content/drive/MyDrive/melisa-ile-biyoinformatik'
ge = pd.read_csv(f'{KLASOR}/CRISPRGeneEffect.csv', index_col=0)
model = pd.read_csv(f'{KLASOR}/Model.csv', index_col=0)
print('gene effect:', ge.shape, '| model:', model.shape)
print(ge.iloc[:3, :4])

Mounted at /content/drive
gene effect: (1208, 18531) | model: (2154, 48)
            A1BG (1)  A1CF (29974)   A2M (2)  A2ML1 (144568)
ACH-000029  0.000151      0.000541 -0.046740        0.040447
ACH-000030  0.169029     -0.182773 -0.023095        0.068845
ACH-000074 -0.087069     -0.054984  0.211763        0.421891


## 2. Chronos skoru nedir

Matristeki her sayı bir **Chronos** skorudur ve ölçeği iki referans noktasına göre kurulmuştur:

- **0 civarı**: geni nakavt etmek hücreye bir şey yapmadı. Sıfır noktası, hiçbir geni hedeflemeyen kontrol sgRNA'ların medyanıdır.
- **−1 civarı**: nakavt hücreyi öldürdü ya da çoğalmasını ciddi biçimde durdurdu. Bu nokta da bir referanstır: bilinen pan-esansiyel genlerin (ribozom, proteazom, RNA polimeraz — her hücrede yaşamsal) medyanı −1'e denk gelecek şekilde ölçeklenmiştir.
- **Pozitif değerler**: nakavt hücreye çoğalma avantajı sağladı; tümör baskılayıcı genlerde görülür.

Yani skor mutlak bir ölçü değil, iki referans arasına yerleştirilmiş göreli bir konumdur. Bunu bilmek yorumun yarısıdır.

Aşağıda dört gen üzerinden ölçeği görüyoruz: RPL23A ve PSMA1 (ribozom ve proteazom alt birimleri, pan-esansiyel beklenir), OR2T1 (koku alma reseptörü, hücre hatlarında ifade edilmez, etkisiz beklenir) ve KRAS (yalnız bazı hatlarda kritik olan onkogen). Sütun adları `SEMBOL (EntrezID)` biçiminde olduğu için sembole göre arama yapan küçük bir yardımcı yazıyoruz.

In [2]:
def sutun(sembol):
    eslesen = [c for c in ge.columns if c.split(' ')[0] == sembol]
    return eslesen[0] if eslesen else None

for g in ['RPL23A', 'PSMA1', 'OR2T1', 'KRAS']:
    c = sutun(g)
    if c:
        s = ge[c].dropna()
        print(f'{g:8s} medyan: {s.median():6.2f} | min: {s.min():6.2f} | maks: {s.max():6.2f}')

RPL23A   medyan:  -2.43 | min:  -4.00 | maks:  -0.84
PSMA1    medyan:  -2.36 | min:  -3.45 | maks:  -0.73
KRAS     medyan:  -0.52 | min:  -4.46 | maks:   0.26


## 3. Künyeyi tanımak

`Model.csv` her hücre hattının kimliğini taşır. Kullanacağımız sütunlar: `CellLineName` (hattın adı), `OncotreeLineage` (doku kökeni — Lung, Breast gibi), `OncotreePrimaryDisease` (ayrıntılı tanı). İki tablo `ModelID` (ACH-XXXXXX) üzerinden eşleşir; birinci serideki "ad etikettir, kimlik esastır" kuralının bu projedeki karşılığı budur. Her hücre hattının CRISPR ekranı yoktur, bu yüzden iki tablonun kesişimini alıyoruz.

In [3]:
print(model[['CellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease']].head())

ortak = ge.index.intersection(model.index)
print('\nekranı olan hücre hattı:', len(ortak))
print('\nen çok hattı olan 10 doku:')
print(model.loc[ortak, 'OncotreeLineage'].value_counts().head(10))

           CellLineName       OncotreeLineage     OncotreePrimaryDisease
ModelID                                                                 
ACH-000001  NIH:OVCAR-3  Ovary/Fallopian Tube   Ovarian Epithelial Tumor
ACH-000002        HL-60               Myeloid     Acute Myeloid Leukemia
ACH-000003        CACO2                 Bowel  Colorectal Adenocarcinoma
ACH-000004          HEL               Myeloid     Acute Myeloid Leukemia
ACH-000005   HEL 92.1.7               Myeloid     Acute Myeloid Leukemia

ekranı olan hücre hattı: 1208

en çok hattı olan 10 doku:
OncotreeLineage
Lung                    126
Lymphoid                 96
CNS/Brain                91
Head and Neck            77
Skin                     75
Esophagus/Stomach        69
Bowel                    63
Ovary/Fallopian Tube     59
Bone                     58
Breast                   53
Name: count, dtype: int64


## 4. Ders alt kümesi

Tam matris 400 MB'ın üzerinde; her defterde indirmek pratik değil. Alt kümeyi iki kuralla çıkarıyoruz:

1. **Hücre hatları**: hepsi kalıyor. Bu projenin bütün soruları hatları karşılaştırmak üzerine kurulu; hat kaybetmek soru kaybetmektir.
2. **Genler**: önce eksik değeri olan sütunları atıyoruz, sonra skorları arasında en çok değişkenlik gösteren 4.000 geni alıyoruz. Gerekçe: bir genin skoru bütün hatlarda aynıysa (hep 0 ya da hep −1 civarı) hat karşılaştırması için bilgi taşımaz. Üstüne, ölçeği anlatırken gerekecek 500 pan-esansiyel referans geni ekliyoruz — en düşük medyanlı genler.

Künyeyi de kesişime indirip iki dosya olarak kaydediyoruz. Dosyalar Drive'a yazılıyor, böylece Colab oturumu kapansa da kalıyorlar.

In [4]:
tam = ge.loc[ortak].dropna(axis=1)
print('eksiksiz gen:', tam.shape[1])

std = tam.std().sort_values(ascending=False)
medyan = tam.median()
degisken = list(std.head(4000).index)
pan = [g for g in medyan.sort_values().head(500).index if g not in degisken]
secili = degisken + pan

alt = tam[secili].copy()
kunye = model.loc[ortak, ['CellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease']]
print('alt küme:', alt.shape, f'({len(degisken)} değişken + {len(pan)} pan-esansiyel)')

import os
alt.to_csv(f'{KLASOR}/depmap_ders_skorlar.csv')
kunye.to_csv(f'{KLASOR}/depmap_ders_kunye.csv')
for f in ['depmap_ders_skorlar.csv', 'depmap_ders_kunye.csv']:
    print(f'{f}: {os.path.getsize(f"{KLASOR}/{f}")/1e6:.1f} MB')

eksiksiz gen: 17087
alt küme: (1208, 4000) (4000 değişken + 0 pan-esansiyel)
depmap_ders_skorlar.csv: 95.2 MB
depmap_ders_kunye.csv: 0.1 MB


## Kendin dene

Üç görev. Birincisi: tam matrisin boyutlarını, ekranı olan hücre hattı sayısını ve en çok hattı olan üç dokuyu not edin. İkincisi: dört genin medyan skorlarını yazın ve tek cümleyle açıklayın — RPL23A ile OR2T1 arasındaki fark Chronos ölçeğinin hangi iki referans noktasına karşılık geliyor? Üçüncüsü: Drive'a yazılan iki CSV dosyasını Zenodo'ya yeni bir kayıt olarak yükleyin (başlıkta DepMap sürümü, açıklamada Broad Institute atfı, lisans CC-BY), DOI'yi alın. 4.2'den itibaren bütün defterler veriyi o adresten okuyacak.

In [5]:
# Ek: nötr gen örneği ve daha küçük dosya
rng = np.random.default_rng(0)
kalan = [g for g in tam.columns if g not in secili]
notr = list(rng.choice(kalan, size=1000, replace=False))
alt = tam[secili + notr].round(3)
alt.to_csv(f'{KLASOR}/depmap_ders_skorlar.csv')
print('yeni alt küme:', alt.shape,
      '| %.1f MB' % (os.path.getsize(f'{KLASOR}/depmap_ders_skorlar.csv')/1e6))

yeni alt küme: (1208, 5000) | 40.5 MB
